In [0]:
from pyspark.sql.functions import(
    trim,
    col
)
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

##### Importing from utils and validation_utils package

In [0]:
from utils.custom_utils import Transformations
from validation_utils.test_utils import Validations
tr_obj = Transformations()
va_obj = Validations()

#### Reading bronze table

In [0]:
df = spark.table("lakehouse.bronze.cust_az12")

### Silver layer transformations

#### 1. Splitting the CID
- The cid as it is cannot be used for joins so, first separate this column so that the values match with the column of crm_customers
- After removing the first three characters, the resulting column can be used for joins with crm_customers using the customer_number column

In [0]:
df = df.withColumn(
    "CID",
    F.when(col("CID").startswith("NAS"),
           F.substring(col("CID"), 4, F.length(col("CID")))
           ).otherwise(col("CID"))
)

#### 2. Renaming Columns

In [0]:
rename_config = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name, new_name in rename_config.items():
    df = df.withColumnRenamed(old_name, new_name)

#### 3. Removing records with no customer_number
- This table doesnot have any nulls in customer_number

In [0]:
va_obj.null_check(df = df, primary_col = "customer_number")

#### 4. Removing customer_number duplicates
- This table doesnot have any customer_number duplicates

In [0]:
va_obj.duplicate_check(df = df, dedup_cols = ["customer_number"], cdc = "birth_date")

#### 5. Trimming

In [0]:
df = tr_obj.trim_columns(df)

#### 6. Normalization

In [0]:
df = df.withColumn(
    "gender",
    F.when(F.upper(col("gender")).isin("M", "MALE"), "Male")
    .when(F.upper(col("gender")).isin("F", "FEMALE"), "Female")
    .otherwise("n/a")
)

#### 7. Cleaning birth_date
- If the birth_date > curren_date() set it to null

In [0]:
df = df.withColumn(
    "birth_date",
    F.when(col("birth_date") > F.current_date(), None)
    .otherwise(col("birth_date"))
)

#### DataFrame sanity check

In [0]:
df.limit(10).display()

#### 8. Applying upsert(SCD type 1)

In [0]:
target_table = "lakehouse.silver.erp_customers"
if spark.catalog.tableExists(target_table):
    tr_obj.upsert(
        spark = spark,
        df = df,
        key_cols = ["customer_number"],
        table = "erp_customers",
        cdc = "birth_date"
    )
else:
    (
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
    )

#### Silver table sanity check

In [0]:
%sql
select *
from lakehouse.silver.erp_customers
limit 10